# EmoCraft: facial emotion recognition (FER) with image and video prediction

A CNN trained on a FER-style image dataset, plus image prediction and video prediction. RAVDESS speech-video archives are an optional external video sample/evaluation set; they do not replace the image training data.

**What this version fixes compared with the original notebook**

- The test set is no longer used for validation. Validation comes from `val/` if it exists, otherwise from a stratified 10% of `train/`. `test/` is used once, at the end.
- The test generator no longer shuffles, so the confusion matrix lines up with the true labels. Metrics are computed from the same ordered predictions.
- All dataset classes are used (including `disgust`) and the output layer size is taken from the data, not hard-coded.
- Class names come from one source of truth (sorted folder names) and are saved with the model.
- No hard-coded user paths. Set `DATASET_DIR` or `DATASET_ZIP` in the configuration cell (or upload a zip in Colab).
- Class weights, early stopping, learning-rate reduction, best-checkpoint saving and a fixed seed.
- Rescaling is part of the model, so training, evaluation and prediction all use the same preprocessing.
- Image and video prediction detect faces, validate input, sample video by time, and return a clip summary.

Run the cells from top to bottom.

- Optional RAVDESS support reads the downloaded `Video_Speech_Actor_*.zip` files, extracts only video-only speech clips, maps emotion labels to the trained model, and selects a clip for the video demo.


In [1]:
# 1. Dependencies. Only installs what is missing (Colab already ships with all of these).
import importlib.util, subprocess, sys

REQUIRED = {"tensorflow": "tensorflow", "cv2": "opencv-python-headless", "sklearn": "scikit-learn",
            "matplotlib": "matplotlib", "PIL": "pillow", "numpy": "numpy"}
missing = [pip for mod, pip in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("Done. If an import below fails, restart the runtime/kernel and run again.")
else:
    print("All required packages are already installed.")

All required packages are already installed.


In [2]:
# 2. Imports and environment report
import os, json, time, zipfile, shutil, random
from pathlib import Path
from collections import Counter

import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, UnidentifiedImageError
import tensorflow as tf
import keras
from keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, balanced_accuracy_score,
                             classification_report, confusion_matrix, f1_score, log_loss)

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except (ImportError, ValueError):
    IN_COLAB = False

gpus = tf.config.list_physical_devices("GPU")
print("Runtime:", "Colab" if IN_COLAB else "local")
for name, mod in [("python", None), ("tensorflow", tf), ("keras", keras), ("numpy", np), ("opencv", cv2)]:
    print(f"  {name:<11}", sys.version.split()[0] if mod is None else mod.__version__)
print("  GPUs       ", [g.name for g in gpus] if gpus else "none (training will run on CPU)")

Runtime: local
  python      3.14.6
  tensorflow  2.22.0-rc0
  keras       3.16.0.dev2026103003
  numpy       2.4.6
  opencv      5.0.0
  GPUs        none (training will run on CPU)


In [3]:
# 3. Configuration. Edit values here, or override with EMOCRAFT_* environment variables.
def _env(name, default, cast=str):
    v = os.environ.get(name)
    return default if v in (None, "") else cast(v)

# --- Data --------------------------------------------------------------------------------
DATASET_DIR = _env("EMOCRAFT_DATASET_DIR", None)   # folder that contains train/ and test/ (and optionally val/)
DATASET_ZIP = _env("EMOCRAFT_DATASET_ZIP", None)   # or a .zip of that folder; in Colab, leave both None to upload
RAVDESS_ZIP_DIR = Path(_env("EMOCRAFT_RAVDESS_DIR", str(Path.cwd() / "RAVDESS"))).expanduser()
# Folder holding the Zenodo Video_Speech_Actor_*.zip files; place RAVDESS/ beside this notebook or set the env var above.
WORK_DIR = Path(_env("EMOCRAFT_WORK_DIR", "/content/emocraft" if IN_COLAB else "emocraft_work")).resolve()
CLASSES_TO_USE = None                              # None = every class folder; or e.g. ["angry", "happy", "sad"]
VERIFY_IMAGES = True                               # open every image once and drop unreadable files

# --- Model / training --------------------------------------------------------------------
IMAGE_SIZE = 48
COLOR_MODE = "grayscale"                           # "grayscale" (FER images are grayscale) or "rgb"
BATCH_SIZE = 64
EPOCHS = _env("EMOCRAFT_EPOCHS", 40, int)
LEARNING_RATE = 1e-3
VAL_FRACTION = 0.10                                # used only when the dataset has no val/ folder
SEED = 42
EARLY_STOP_PATIENCE = 8
LR_REDUCE_PATIENCE = 3
BN_MOMENTUM = _env("EMOCRAFT_BN_MOMENTUM", 0.99, float)   # Keras default; see the note after training

# --- Prediction ----------------------------------------------------------------------------
SAMPLE_IMAGE_PATH = _env("EMOCRAFT_SAMPLE_IMAGE", None)   # path to a test image (Colab: upload when asked)
SAMPLE_VIDEO_PATH = _env("EMOCRAFT_SAMPLE_VIDEO", None)   # path to a test video
FACE_DETECTION = dict(scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
PRECROPPED_MAX_SIDE = 96                           # images this small are treated as already-cropped faces

CHANNELS = 1 if COLOR_MODE == "grayscale" else 3
ARTIFACT_DIR = WORK_DIR / "artifacts"
WORK_DIR.mkdir(parents=True, exist_ok=True); ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); keras.utils.set_random_seed(SEED)
print("WORK_DIR    :", WORK_DIR)
print("ARTIFACT_DIR:", ARTIFACT_DIR)

WORK_DIR    : C:\Users\aweso\OneDrive\Desktop\sun\emocraft_work
ARTIFACT_DIR: C:\Users\aweso\OneDrive\Desktop\sun\emocraft_work\artifacts


## Datasets

The CNN still needs an image dataset in `train/<class>/...` and `test/<class>/...` format (optionally `val/`). Use FER2013 for the direct 48x48 grayscale fit. RAVDESS is a separate video set: it is used below for a sample-video run and optional external evaluation, not as a substitute for the image training split.

For this computer, the downloaded archives should be in `DL/RAVDESS/`. If you move them, set `EMOCRAFT_RAVDESS_DIR` or change `RAVDESS_ZIP_DIR` in the configuration cell. In Colab, upload/copy the actor ZIP files to a `RAVDESS/` folder.


In [4]:
# 4. Locate / extract the dataset
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def _has_splits(p):
    return (p / "train").is_dir() and (p / "test").is_dir()

def find_dataset_root(base, max_depth=3):
    """Return the folder under `base` that contains train/ and test/ (shallowest first)."""
    base = Path(base)
    if _has_splits(base):
        return base
    for depth in range(1, max_depth + 1):
        hits = sorted(p.parent for p in base.glob("/".join(["*"] * depth) + "/train")
                      if p.is_dir() and "__MACOSX" not in p.parts and _has_splits(p.parent))
        if hits:
            return hits[0]
    return None

def extract_zip(zip_path, target_dir):
    zip_path, target_dir = Path(zip_path), Path(target_dir)
    marker = target_dir / ".extracted_ok"
    if marker.exists():
        print(f"Re-using previous extraction in {target_dir}")
        return target_dir
    if target_dir.exists():
        shutil.rmtree(target_dir)
    target_dir.mkdir(parents=True)
    print(f"Extracting {zip_path.name} -> {target_dir}")
    with zipfile.ZipFile(zip_path) as z:
        members = [m for m in z.namelist() if not m.startswith("__MACOSX") and "/._" not in m and not Path(m).name.startswith("._")]
        z.extractall(target_dir, members=members)
    marker.write_text("ok")
    return target_dir

def resolve_dataset_dir():
    if DATASET_DIR:
        root = find_dataset_root(DATASET_DIR)
        if root is None:
            raise FileNotFoundError(f"DATASET_DIR='{DATASET_DIR}' does not exist or has no 'train' and 'test' sub-folders. "
                                    "Point it at the folder that contains train/ and test/.")
        return root
    zip_path = DATASET_ZIP
    if not zip_path and IN_COLAB:
        from google.colab import files
        print("Upload the dataset .zip (folder with train/ and test/):")
        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError("No file was uploaded.")
        zip_path = next(iter(uploaded))
    if not zip_path:
        raise FileNotFoundError("No dataset configured. Set DATASET_DIR (folder with train/ and test/) or DATASET_ZIP in the "
                                "configuration cell. In Colab you can also leave both as None and upload a zip when asked.")
    if not Path(zip_path).is_file():
        raise FileNotFoundError(f"DATASET_ZIP not found: {zip_path}")
    extracted = extract_zip(zip_path, WORK_DIR / "dataset" / Path(zip_path).stem)
    root = find_dataset_root(extracted)
    if root is None:
        raise FileNotFoundError(f"The zip was extracted to {extracted} but no folder with train/ and test/ was found inside it.")
    return root

DATASET_ROOT = resolve_dataset_dir()
print("Dataset root:", DATASET_ROOT)

FileNotFoundError: No dataset configured. Set DATASET_DIR (folder with train/ and test/) or DATASET_ZIP in the configuration cell. In Colab you can also leave both as None and upload a zip when asked.

In [ ]:
# 5. Scan every split, validate it and build the class list (single source of truth for labels)
def list_class_dirs(split_dir):
    return sorted(d.name for d in split_dir.iterdir() if d.is_dir() and not d.name.startswith((".", "_")))

def readable(path):
    try:
        with Image.open(path) as im:
            im.verify()
        return True
    except Exception:
        return False

def scan_split(split_dir):
    """Return {class_name: [image paths]}; hidden/non-image files are ignored, unreadable images are dropped."""
    out, skipped_other, bad = {}, 0, []
    for cls in list_class_dirs(split_dir):
        files = []
        for p in sorted((split_dir / cls).iterdir()):
            if not p.is_file():
                continue
            if p.name.startswith(".") or p.suffix.lower() not in IMAGE_EXTS:
                skipped_other += 1
            elif VERIFY_IMAGES and not readable(p):
                bad.append(p)
            else:
                files.append(p)
        out[cls] = files
    if bad:
        print(f"WARNING: {len(bad)} unreadable image(s) excluded from '{split_dir.name}', e.g. {bad[0]}")
    if skipped_other:
        print(f"Note: {skipped_other} non-image/hidden file(s) ignored in '{split_dir.name}'")
    return out

split_names = [s for s in ("train", "val", "test") if (DATASET_ROOT / s).is_dir()]
scans = {s: scan_split(DATASET_ROOT / s) for s in split_names}

# class folders must be identical across splits
ref = set(scans["train"])
problems = []
for s in split_names:
    if set(scans[s]) != ref:
        problems.append(f"  '{s}' differs from 'train': missing {sorted(ref - set(scans[s]))}, unexpected {sorted(set(scans[s]) - ref)}")
if problems:
    raise ValueError("Class folders do not match across splits:\n" + "\n".join(problems) +
                     "\nRename/move the folders so train, test (and val) contain identical class names.")

all_classes = sorted(ref)
if CLASSES_TO_USE:
    unknown = sorted(set(CLASSES_TO_USE) - set(all_classes))
    if unknown:
        raise ValueError(f"CLASSES_TO_USE contains classes that are not in the dataset: {unknown}. Available: {all_classes}")
    CLASS_NAMES = sorted(CLASSES_TO_USE)
    print("NOTE: classes deliberately excluded via CLASSES_TO_USE:", sorted(set(all_classes) - set(CLASS_NAMES)))
else:
    CLASS_NAMES = all_classes
NUM_CLASSES = len(CLASS_NAMES)
CLASS_TO_ID = {c: i for i, c in enumerate(CLASS_NAMES)}
empty = [(s, c) for s in split_names for c in CLASS_NAMES if not scans[s][c]]
if empty:
    raise ValueError(f"Empty class folders (no readable images): {empty}")
if NUM_CLASSES < 2:
    raise ValueError("Need at least two classes.")

print(f"Classes used ({NUM_CLASSES}): {CLASS_NAMES}")
print("Label mapping:", CLASS_TO_ID)
print(f"\n{'class':<12}" + "".join(f"{s:>8}" for s in split_names))
for c in CLASS_NAMES:
    print(f"{c:<12}" + "".join(f"{len(scans[s][c]):>8}" for s in split_names))
print(f"{'total':<12}" + "".join(f"{sum(len(scans[s][c]) for c in CLASS_NAMES):>8}" for s in split_names))
train_counts = np.array([len(scans["train"][c]) for c in CLASS_NAMES])
print(f"\nTrain class imbalance (largest / smallest class): {train_counts.max() / train_counts.min():.1f}x")

In [ ]:
# 6. Look at the data: class balance and one example per class (train split)
fig, ax = plt.subplots(1, len(split_names), figsize=(5 * len(split_names), 3.6), squeeze=False)
for a, s in zip(ax[0], split_names):
    counts = [len(scans[s][c]) for c in CLASS_NAMES]
    a.bar(CLASS_NAMES, counts, color="steelblue"); a.set_title(f"{s}: images per class")
    a.tick_params(axis="x", rotation=45)
    for i, n in enumerate(counts):
        a.text(i, n, str(n), ha="center", va="bottom", fontsize=8)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, NUM_CLASSES, figsize=(2.2 * NUM_CLASSES, 2.6), squeeze=False)
for a, c in zip(ax[0], CLASS_NAMES):
    with Image.open(scans["train"][c][0]) as im:
        a.imshow(im.convert("L"), cmap="gray"); a.set_title(c); a.axis("off")
plt.tight_layout(); plt.show()

## Data pipeline

Images are decoded to `float32` in the range 0 to 255 and rescaled **inside the model**, so any code that calls the model gets the same preprocessing. The validation set is separate from the test set. The test set is never shuffled.

In [ ]:
# 7. Validation split, class weights and tf.data pipelines
def flatten(scan):
    paths, labels = [], []
    for c in CLASS_NAMES:
        paths += scan[c]; labels += [CLASS_TO_ID[c]] * len(scan[c])
    return paths, np.array(labels, dtype=np.int32)

train_all_paths, train_all_labels = flatten(scans["train"])
test_paths, test_labels = flatten(scans["test"])

if "val" in scans:
    fit_paths, fit_labels = train_all_paths, train_all_labels
    val_paths, val_labels = flatten(scans["val"])
    VAL_SOURCE = "provided 'val/' folder"
else:
    idx_fit, idx_val = train_test_split(np.arange(len(train_all_paths)), test_size=VAL_FRACTION,
                                        stratify=train_all_labels, random_state=SEED)
    fit_paths = [train_all_paths[i] for i in idx_fit]; fit_labels = train_all_labels[idx_fit]
    val_paths = [train_all_paths[i] for i in idx_val]; val_labels = train_all_labels[idx_val]
    VAL_SOURCE = f"stratified {int(VAL_FRACTION * 100)}% of train/ (seed {SEED})"
print("Validation source:", VAL_SOURCE)
print(f"{'class':<12}{'fit':>8}{'val':>8}{'test':>8}")
for c in CLASS_NAMES:
    i = CLASS_TO_ID[c]
    print(f"{c:<12}{(fit_labels == i).sum():>8}{(val_labels == i).sum():>8}{(test_labels == i).sum():>8}")
print(f"{'total':<12}{len(fit_labels):>8}{len(val_labels):>8}{len(test_labels):>8}")

# balanced class weights from the fitting data only: n / (k * count)
fit_counts = np.bincount(fit_labels, minlength=NUM_CLASSES)
CLASS_WEIGHTS = {i: float(len(fit_labels) / (NUM_CLASSES * fit_counts[i])) for i in range(NUM_CLASSES)}
print("\nClass weights:", {CLASS_NAMES[i]: round(w, 2) for i, w in CLASS_WEIGHTS.items()})

def resize_for_model(img):
    """Shared by training and prediction so both see identically resized pixels. img: HxWxC float32, 0..255."""
    return tf.image.resize(img, (IMAGE_SIZE, IMAGE_SIZE), method="bilinear", antialias=True)

def _decode(path, label):
    img = tf.io.decode_image(tf.io.read_file(path), channels=CHANNELS, expand_animations=False)
    img = resize_for_model(tf.cast(img, tf.float32))
    img.set_shape((IMAGE_SIZE, IMAGE_SIZE, CHANNELS))
    return img, label

def make_ds(paths, labels, training):
    ds = tf.data.Dataset.from_tensor_slices(([str(p) for p in paths], labels))
    if training:
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(_decode, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(fit_paths, fit_labels, training=True)
val_ds = make_ds(val_paths, val_labels, training=False)
test_ds = make_ds(test_paths, test_labels, training=False)     # never shuffled: order == test_labels

xb, yb = next(iter(train_ds))
print(f"\nOne training batch: x{tuple(xb.shape)} {xb.dtype.name} values {xb.numpy().min():.0f}..{xb.numpy().max():.0f}; "
      f"y{tuple(yb.shape)} {yb.dtype.name}")

## Model

Same CNN idea as the original (4 convolution blocks + dense layer), with:
rescaling and augmentation as layers (augmentation is only active during training), `softmax` output sized from the data, and a configurable BatchNorm momentum.

In [ ]:
# 8. Model
def build_model(num_classes, image_size, channels, seed, bn_momentum=0.99):
    inp = keras.Input((image_size, image_size, channels), name="image_0_255")
    x = layers.Rescaling(1.0 / 255, name="rescale")(inp)
    x = layers.RandomFlip("horizontal", seed=seed)(x)
    x = layers.RandomRotation(15 / 360, fill_mode="nearest", seed=seed)(x)
    x = layers.RandomTranslation(0.1, 0.1, fill_mode="nearest", seed=seed)(x)
    x = layers.RandomZoom(0.2, fill_mode="nearest", seed=seed)(x)
    for filters in (32, 64, 128, 256):
        x = layers.Conv2D(filters, 3, activation="relu")(x)
        x = layers.BatchNormalization(momentum=bn_momentum)(x)
        x = layers.MaxPooling2D(2)(x)
        x = layers.Dropout(0.25)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(512, activation="relu")(x)
    x = layers.BatchNormalization(momentum=bn_momentum)(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(num_classes, activation="softmax", name="emotion_probs")(x)
    return keras.Model(inp, out, name="emocraft_cnn")

model = build_model(NUM_CLASSES, IMAGE_SIZE, CHANNELS, SEED, BN_MOMENTUM)
model.compile(optimizer=keras.optimizers.Adam(LEARNING_RATE),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
assert model.output_shape[-1] == NUM_CLASSES
print(f"Output units: {model.output_shape[-1]} (classes: {NUM_CLASSES}). Parameters: {model.count_params():,}")

## Training

The best epoch is chosen by **validation** loss. The test set is not touched here.

In [ ]:
# 9. Train
CHECKPOINT_PATH = ARTIFACT_DIR / "best_checkpoint.keras"
callbacks = [
    keras.callbacks.ModelCheckpoint(CHECKPOINT_PATH, monitor="val_loss", save_best_only=True),
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=EARLY_STOP_PATIENCE, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=LR_REDUCE_PATIENCE, verbose=1),
]
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks,
                    class_weight=CLASS_WEIGHTS, shuffle=False, verbose=2)   # tf.data already shuffles

hist = history.history
BEST_EPOCH = int(np.argmin(hist["val_loss"]))
print(f"\nTrained for {len(hist['loss'])} epoch(s) (limit {EPOCHS}). Best validation loss at epoch {BEST_EPOCH + 1}: "
      f"val_loss={hist['val_loss'][BEST_EPOCH]:.4f}, val_accuracy={hist['val_accuracy'][BEST_EPOCH]:.4f}")

# Sanity check (train + validation only): accuracy in inference mode, i.e. without augmentation/dropout.
_, train_eval_acc = model.evaluate(train_ds, verbose=0)
_, val_eval_acc = model.evaluate(val_ds, verbose=0)
print(f"Inference-mode accuracy of the selected model: train={train_eval_acc:.3f}, validation={val_eval_acc:.3f} "
      f"(training-mode accuracy at that epoch: {hist['accuracy'][BEST_EPOCH]:.3f})")
if train_eval_acc < 0.5 * hist["accuracy"][BEST_EPOCH]:
    print("WARNING: inference-mode accuracy on the training images is far below the training-mode accuracy. "
          "With few training steps per epoch, BatchNorm's moving statistics can lag behind the weights. "
          "Use more data/epochs, or try BN_MOMENTUM = 0.9 in the configuration cell.")

In [ ]:
# 10. Training curves
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for a, key, title in ((ax[0], "accuracy", "Accuracy"), (ax[1], "loss", "Loss")):
    a.plot(hist[key], label=f"train {key}"); a.plot(hist[f"val_{key}"], label=f"validation {key}")
    a.axvline(BEST_EPOCH, ls="--", c="gray", lw=1, label=f"best epoch ({BEST_EPOCH + 1})")
    a.set_title(title); a.set_xlabel("epoch"); a.set_ylabel(key); a.legend()
plt.tight_layout(); plt.show()

In [ ]:
# 11. Save the model together with everything needed to use it correctly later
MODEL_PATH = ARTIFACT_DIR / "emocraft_cnn.keras"
CLASS_MAP_PATH = ARTIFACT_DIR / "class_mapping.json"
PREPROC_PATH = ARTIFACT_DIR / "preprocessing_config.json"

model.save(MODEL_PATH)
CLASS_MAP_PATH.write_text(json.dumps({"class_names": CLASS_NAMES, "class_to_id": CLASS_TO_ID}, indent=2))
PREPROCESS_CONFIG = {"image_size": IMAGE_SIZE, "color_mode": COLOR_MODE, "channels": CHANNELS,
                     "input_range": "0..255 (the model rescales by 1/255 internally)",
                     "resize": "tf.image.resize bilinear, antialias=True"}
PREPROC_PATH.write_text(json.dumps(PREPROCESS_CONFIG, indent=2))
for p in (CHECKPOINT_PATH, CLASS_MAP_PATH, MODEL_PATH, PREPROC_PATH):
    print(f"saved: {p}  ({p.stat().st_size / 1024:.0f} KB)")

## Final evaluation on the test set

The test set is evaluated once, with the final selected model. Predictions come from a non-shuffled dataset, so `test_labels[i]` is the true label of prediction `i`.

In [ ]:
# 12. Test-set evaluation
test_probs = model.predict(test_ds, verbose=0)
y_true = test_labels
y_pred = test_probs.argmax(axis=1)
assert len(y_pred) == len(y_true)
label_ids = list(range(NUM_CLASSES))

acc = accuracy_score(y_true, y_pred)
bal_acc = balanced_accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro", labels=label_ids, zero_division=0)
ce = log_loss(y_true, test_probs, labels=label_ids)
majority_class = int(np.bincount(fit_labels).argmax())
majority_acc = float((y_true == majority_class).mean())

print(f"Test images            : {len(y_true)}")
print(f"Accuracy               : {acc:.4f}")
print(f"Balanced accuracy      : {bal_acc:.4f}")
print(f"Macro-F1               : {macro_f1:.4f}")
print(f"Cross-entropy loss     : {ce:.4f}")
print(f"Reference: always predicting '{CLASS_NAMES[majority_class]}' (most common training class) would score {majority_acc:.4f} accuracy here.\n")
print(classification_report(y_true, y_pred, labels=label_ids, target_names=CLASS_NAMES, digits=3, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=label_ids)
fig, ax = plt.subplots(1, 2, figsize=(14, 5.5))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax[0], cmap="Blues", colorbar=False, xticks_rotation=45)
ax[0].set_title("Confusion matrix (counts)")
ConfusionMatrixDisplay(cm / cm.sum(axis=1, keepdims=True), display_labels=CLASS_NAMES).plot(
    ax=ax[1], cmap="Blues", colorbar=False, xticks_rotation=45, values_format=".2f")
ax[1].set_title("Confusion matrix (row-normalised: recall per true class)")
plt.tight_layout(); plt.show()

report = {
    "n_test": int(len(y_true)), "accuracy": acc, "balanced_accuracy": bal_acc, "macro_f1": macro_f1,
    "cross_entropy": ce, "majority_class_baseline_accuracy": majority_acc,
    "class_names": CLASS_NAMES, "validation_source": VAL_SOURCE,
    "epochs_trained": len(hist["loss"]), "best_epoch": BEST_EPOCH + 1, "bn_momentum": BN_MOMENTUM,
    "per_class": classification_report(y_true, y_pred, labels=label_ids, target_names=CLASS_NAMES, output_dict=True, zero_division=0),
    "confusion_matrix_rows_true_cols_pred": confusion_matrix(y_true, y_pred, labels=label_ids).tolist(),
}
(ARTIFACT_DIR / "evaluation_report.json").write_text(json.dumps(report, indent=2))
print("Saved", ARTIFACT_DIR / "evaluation_report.json")

## Prediction on images and videos

The predictor wraps the model, class names and preprocessing. Faces are found with OpenCV's Haar cascade (frontal faces only), cropped, converted the same way as the training images, and classified. If no face is found in an image, the whole image is classified and the result says so.

In [ ]:
# 13. Predictor: face detection + preprocessing + model
class Predictor:
    def __init__(self, model, class_names, preprocess_config, face_params):
        self.model, self.class_names, self.cfg, self.face_params = model, list(class_names), preprocess_config, face_params
        cascade = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
        self.detector = cv2.CascadeClassifier(cascade)
        self.face_detection_available = not self.detector.empty()
        if not self.face_detection_available:
            print("WARNING: could not load OpenCV's face cascade; whole images/frames will be classified instead.")

    def detect_faces(self, rgb):
        """Return boxes (x, y, w, h), largest first."""
        if not self.face_detection_available:
            return []
        gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
        boxes = self.detector.detectMultiScale(gray, **self.face_params)
        return sorted((tuple(int(v) for v in b) for b in boxes), key=lambda b: b[2] * b[3], reverse=True)

    def _prepare(self, rgb_uint8):
        x = cv2.cvtColor(rgb_uint8, cv2.COLOR_RGB2GRAY)[..., None] if self.cfg["channels"] == 1 else rgb_uint8
        return resize_for_model(tf.cast(x, tf.float32)).numpy()

    def predict_faces(self, rgb_crops, batch_size=64):
        """rgb_crops: list of HxWx3 uint8 arrays -> list of result dicts (emotion, confidence, probabilities)."""
        if not rgb_crops:
            return []
        batch = np.stack([self._prepare(c) for c in rgb_crops])
        # direct calls (not model.predict) avoid TensorFlow retracing warnings for small, varying batch sizes
        probs = np.concatenate([self.model(batch[i:i + batch_size], training=False).numpy()
                                for i in range(0, len(batch), batch_size)])
        out = []
        for p in probs:
            k = int(np.argmax(p))
            out.append({"emotion": self.class_names[k], "confidence": float(p[k]),
                        "probabilities": {n: round(float(v), 4) for n, v in zip(self.class_names, p)}})
        return out

    def predict_face(self, rgb_crop):
        return self.predict_faces([rgb_crop])[0]

predictor = Predictor(model, CLASS_NAMES, PREPROCESS_CONFIG, FACE_DETECTION)
print("Predictor ready. Face detection:", "on" if predictor.face_detection_available else "off")

In [ ]:
# 14. Image prediction
def load_rgb_image(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Image file not found: {path}")
    if not path.is_file():
        raise ValueError(f"Not a file: {path}")
    try:
        with Image.open(path) as im:
            im = ImageOps.exif_transpose(im)          # respect phone-camera rotation
            return np.array(im.convert("RGB"))
    except (UnidentifiedImageError, OSError) as e:
        raise ValueError(f"'{path.name}' could not be read as an image ({type(e).__name__}: {e}). "
                         "Use a valid image file such as .jpg, .png or .bmp.") from e

def predict_image(image_path, predictor, detect_face=True, show=True):
    rgb = load_rgb_image(image_path)
    h, w = rgb.shape[:2]
    boxes, face_box, n_faces = [], None, 0
    if not detect_face:
        crop, note = rgb, "Face detection off; the whole image was classified."
    elif min(h, w) <= PRECROPPED_MAX_SIDE:
        crop, note = rgb, f"Small image ({w}x{h}); treated as an already-cropped face."
    else:
        boxes = predictor.detect_faces(rgb)
        n_faces = len(boxes)
        if boxes:
            x, y, bw, bh = face_box = boxes[0]
            crop = rgb[y:y + bh, x:x + bw]
            note = f"{n_faces} face(s) detected; the largest was classified."
        else:
            crop, note = rgb, "No face detected; the whole image was classified (fallback)."
    result = predictor.predict_face(crop)
    result.update({"image_path": str(image_path), "face_detected": face_box is not None,
                   "n_faces": n_faces, "face_box": face_box, "note": note})
    print(f"Predicted emotion: {result['emotion']}  |  model score: {result['confidence'] * 100:.1f}% (uncalibrated)  |  {note}")
    if show:
        fig, ax = plt.subplots(1, 2, figsize=(9, 4.2))
        ax[0].imshow(rgb); ax[0].axis("off"); ax[0].set_title("input")
        if face_box:
            x, y, bw, bh = face_box
            ax[0].add_patch(plt.Rectangle((x, y), bw, bh, fill=False, ec="lime", lw=2))
        ax[1].imshow(predictor._prepare(crop).squeeze().astype("uint8"), cmap="gray" if CHANNELS == 1 else None)
        ax[1].axis("off"); ax[1].set_title(f"model input -> {result['emotion']} ({result['confidence'] * 100:.0f}%)")
        plt.tight_layout(); plt.show()
    return result

def pick_media(path, kind):
    """Use the configured path; in Colab offer an upload; otherwise skip."""
    if path:
        return path
    if IN_COLAB:
        from google.colab import files
        print(f"Upload a {kind}:")
        up = files.upload()
        return next(iter(up)) if up else None
    print(f"No sample {kind} configured. Set SAMPLE_{kind.upper()}_PATH in the configuration cell.")
    return None

In [ ]:
# 15. Try it on one image
sample_image = pick_media(SAMPLE_IMAGE_PATH, "image")
if sample_image:
    image_result = predict_image(sample_image, predictor)
    print("Per-class scores:", image_result["probabilities"])

In [ ]:
# 16. Input validation examples (these are expected to fail with clear messages)
for bad_path in (WORK_DIR / "does_not_exist.jpg", WORK_DIR):
    try:
        predict_image(bad_path, predictor, show=False)
    except (FileNotFoundError, ValueError) as e:
        print(f"{type(e).__name__}: {e}\n")
not_image = WORK_DIR / "not_an_image.txt"; not_image.write_text("this is not an image")
try:
    predict_image(not_image, predictor, show=False)
except ValueError as e:
    print(f"ValueError: {e}")

In [ ]:
# 17. Video prediction. Frames are sampled by TIME (sample_fps), not by a fixed frame count.
VIDEO_SUMMARY_NOTE = ("Summary of sampled-frame predictions (one face per frame by default); it is not a ground-truth "
                      "emotion for the whole video. Scores are uncalibrated.")
FALLBACK_STEP = 10

def _small(rgb, max_w=320):
    """Downscaled copy for previews; returns (image, scale)."""
    s = min(1.0, max_w / rgb.shape[1])
    return (cv2.resize(rgb, None, fx=s, fy=s, interpolation=cv2.INTER_AREA) if s < 1 else rgb.copy()), s

def _annotate(small, scale, boxes_labels):
    t = small.copy()
    for (x, y, w, h), label in boxes_labels:
        x, y, w, h = (int(v * scale) for v in (x, y, w, h))
        cv2.rectangle(t, (x, y), (x + w, y + h), (0, 255, 0), 2)
        (tw, th), base = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
        ty = y - 4 if y - th - 8 >= 0 else y + th + 6                 # above the box if there is room
        cv2.rectangle(t, (x, ty - th - 3), (x + tw + 4, ty + base), (0, 0, 0), -1)
        cv2.putText(t, label, (x + 2, ty), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 0), 1, cv2.LINE_AA)
    return t

def predict_video(video_path, predictor, sample_fps=2.0, frame_step=None, max_samples=None,
                  all_faces=False, show=True, max_preview=8, verbose=True):
    """Classify the face(s) in sampled frames. Returns a dict with status, video_info, frame_predictions, summary, message."""
    video_path = Path(video_path)
    if not video_path.exists():
        raise FileNotFoundError(f"Video file not found: {video_path}")
    if not video_path.is_file():
        raise ValueError(f"Not a file: {video_path}")
    if frame_step is None and not sample_fps > 0:
        raise ValueError("sample_fps must be > 0")
    if frame_step is not None and frame_step < 1:
        raise ValueError("frame_step must be >= 1")

    cap = cv2.VideoCapture(str(video_path))
    try:
        if not cap.isOpened():
            raise ValueError(f"OpenCV could not open '{video_path.name}' as a video. Check the file and its codec/format.")
        fps = float(cap.get(cv2.CAP_PROP_FPS) or 0)
        fps_ok = np.isfinite(fps) and 1 <= fps <= 1000
        if frame_step is not None:
            step = int(frame_step)
        elif fps_ok:
            step = max(1, int(round(fps / sample_fps)))
        else:
            step = FALLBACK_STEP
            print(f"WARNING: the video does not report a valid frame rate; sampling every {step}th frame instead.")

        records, crops = [], []            # one record per classified face
        pool, pool_stride = [], 1          # decimated preview thumbnails (bounded memory)
        frames_read = sampled = with_face = unreadable = 0
        while True:
            if not cap.grab():
                break
            idx = frames_read; frames_read += 1
            if idx % step:
                continue
            if max_samples is not None and sampled >= max_samples:
                break
            ok, bgr = cap.retrieve()
            sampled += 1
            if not ok or bgr is None:
                unreadable += 1
                continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            boxes = predictor.detect_faces(rgb)
            if not boxes:
                continue
            with_face += 1
            use = boxes if all_faces else boxes[:1]
            first = len(records)
            for b in use:
                x, y, w, h = b
                records.append({"frame": idx, "time_s": round(idx / fps, 2) if fps_ok else None, "box": b})
                crops.append(rgb[y:y + h, x:x + w])
            if (with_face - 1) % pool_stride == 0:
                small, scale = _small(rgb)
                pool.append((first, list(use), small, scale, idx))
                if len(pool) >= 2 * max_preview:
                    pool, pool_stride = pool[::2], pool_stride * 2
    finally:
        cap.release()

    info = {"fps": fps if fps_ok else None, "frame_step": step, "frames_read": frames_read, "frames_sampled": sampled,
            "frames_with_face": with_face, "frames_without_face": sampled - unreadable - with_face,
            "unreadable_sampled_frames": unreadable}
    if sampled == unreadable:
        msg = "No readable frames: the video opened but no frame could be decoded."
        print(msg)
        return {"status": "no_readable_frames", "video_info": info, "frame_predictions": [], "summary": None, "message": msg}
    if not records:
        msg = (f"No faces detected in any of the {sampled} sampled frame(s). Try a clearer, frontal face, a smaller "
               "frame_step / higher sample_fps, or check FACE_DETECTION.")
        print(msg)
        return {"status": "no_faces_detected", "video_info": info, "frame_predictions": [], "summary": None, "message": msg}

    preds = predictor.predict_faces(crops)
    for r, p in zip(records, preds):
        r.update(emotion=p["emotion"], confidence=p["confidence"])
    counts = Counter(r["emotion"] for r in records)
    top, top_n = counts.most_common(1)[0]
    summary = {"n_predictions": len(records), "emotion_counts": dict(counts), "most_frequent_emotion": top,
               "share_of_predictions": top_n / len(records),
               "mean_confidence": float(np.mean([r["confidence"] for r in records])),
               "mean_confidence_of_most_frequent": float(np.mean([r["confidence"] for r in records if r["emotion"] == top])),
               "note": VIDEO_SUMMARY_NOTE}
    if verbose:
        print(f"{len(records)} prediction(s) from {with_face} of {sampled} sampled frame(s) (every {step} frame(s)).")
        print(f"Most frequent predicted emotion: {top} ({summary['share_of_predictions'] * 100:.0f}% of predictions) | "
              f"mean model score: {summary['mean_confidence'] * 100:.1f}%")
        print("Counts:", dict(counts))
        for r in records[:30]:
            t = f"{r['time_s']:.1f}s" if r["time_s"] is not None else f"frame {r['frame']}"
            print(f"  {t:>8}: {r['emotion']:<9} {r['confidence'] * 100:5.1f}%")
        if len(records) > 30:
            print(f"  ... {len(records) - 30} more (see the returned 'frame_predictions')")
        print("Note:", VIDEO_SUMMARY_NOTE)
    if show and pool:
        picks = sorted(set(np.linspace(0, len(pool) - 1, min(max_preview, len(pool))).round().astype(int)))
        fig, ax = plt.subplots(1, len(picks), figsize=(3.4 * len(picks), 3.3), squeeze=False)
        for a_, k in zip(ax[0], picks):
            first, used, small, scale, idx = pool[k]
            labels = [(b, f"{records[first + j]['emotion']} {records[first + j]['confidence'] * 100:.0f}%")
                      for j, b in enumerate(used)]
            a_.imshow(_annotate(small, scale, labels)); a_.axis("off")
            a_.set_title(f"{idx / fps:.1f}s" if fps_ok else f"frame {idx}", fontsize=9)
        plt.tight_layout(); plt.show()
    return {"status": "ok", "video_info": info, "frame_predictions": records, "summary": summary, "message": "ok"}

## RAVDESS video data (optional)

This loader reads only **video-only speech** entries from `Video_Speech_Actor_*.zip` archives. It skips the audio-video duplicate, songs, and emotion folders that do not map to the trained model. It understands the filename fields from the official RAVDESS naming convention. The included download currently has actors 05 and 06 only, so treat any resulting scores as a small pilot, not a broad benchmark.
Source: [RAVDESS on Zenodo](https://zenodo.org/records/1188976). The dataset is released under CC BY-NC-SA 4.0; cite it and follow its license when using or sharing results.


In [ ]:
# Read RAVDESS archive members and extract only compatible video-only speech clips.
RAVDESS_VIDEO_DIR = WORK_DIR / "ravdess_video_only"
RAVDESS_SAMPLE_FPS = 1.0
RAVDESS_VIDEO_DIR.mkdir(parents=True, exist_ok=True)

# Filename emotion IDs: 01 neutral, 02 calm, 03 happy, 04 sad, 05 angry,
# 06 fearful, 07 disgust, 08 surprised. Calm is retained only if the image model has that class.
RAVDESS_EMOTIONS = {
    "01": "neutral", "02": "calm", "03": "happy", "04": "sad",
    "05": "angry", "06": "fear", "07": "disgust", "08": "surprise",
}
RAVDESS_CLASS_ALIASES = {
    "neutral": {"neutral"}, "calm": {"calm"}, "happy": {"happy"}, "sad": {"sad"},
    "angry": {"angry", "anger"}, "fear": {"fear", "fearful"},
    "disgust": {"disgust"}, "surprise": {"surprise", "surprised"},
}

def _ravdess_model_class(canonical_label):
    accepted = RAVDESS_CLASS_ALIASES.get(canonical_label, {canonical_label})
    for model_label in CLASS_NAMES:
        if model_label.strip().lower() in accepted:
            return model_label
    return None

def load_ravdess_video_records(zip_dir=RAVDESS_ZIP_DIR, output_dir=RAVDESS_VIDEO_DIR):
    """Extract labeled modality-02 speech clips. Never extracts arbitrary zip paths."""
    zip_dir, output_dir = Path(zip_dir), Path(output_dir)
    archives = sorted(zip_dir.glob("Video_Speech_Actor_*.zip")) if zip_dir.is_dir() else []
    if not archives:
        print(f"No RAVDESS speech-video archives found in {zip_dir}. This is optional; add Video_Speech_Actor_*.zip files to use it.")
        return []

    records, skipped_unmatched = [], 0
    for archive in archives:
        with zipfile.ZipFile(archive) as zf:
            for info in zf.infolist():
                if info.is_dir() or Path(info.filename).suffix.lower() != ".mp4":
                    continue
                fields = Path(info.filename).stem.split("-")
                if len(fields) != 7:
                    skipped_unmatched += 1
                    continue
                modality, vocal_channel, emotion_id, intensity, statement, repetition, actor = fields
                # 02 = video only; 01 = speech. Do not double-count the audio-video modality.
                if modality != "02" or vocal_channel != "01":
                    continue
                canonical = RAVDESS_EMOTIONS.get(emotion_id)
                model_label = _ravdess_model_class(canonical) if canonical else None
                if model_label is None:
                    skipped_unmatched += 1
                    continue

                actor_dir = output_dir / f"Actor_{actor}"
                actor_dir.mkdir(parents=True, exist_ok=True)
                destination = actor_dir / Path(info.filename).name
                if not destination.is_file() or destination.stat().st_size != info.file_size:
                    with zf.open(info, "r") as source_file, destination.open("wb") as target_file:
                        shutil.copyfileobj(source_file, target_file)
                if destination.stat().st_size != info.file_size:
                    raise IOError(f"Incomplete RAVDESS extraction: {destination}")
                records.append({
                    "path": destination,
                    "label": model_label,
                    "canonical_label": canonical,
                    "actor": actor,
                    "emotion_id": emotion_id,
                    "filename": Path(info.filename).name,
                })

    records.sort(key=lambda row: (row["actor"], row["label"], row["filename"]))
    print(f"RAVDESS archives: {len(archives)} | compatible video-only speech clips: {len(records)} | skipped/unmapped entries: {skipped_unmatched}")
    if records:
        print("RAVDESS clips by model label:", dict(sorted(Counter(r["label"] for r in records).items())))
        print("Actors present:", sorted({r["actor"] for r in records}))
    return records

RAVDESS_RECORDS = load_ravdess_video_records()
RAVDESS_SAMPLE_VIDEO = str(RAVDESS_RECORDS[0]["path"]) if RAVDESS_RECORDS else None


### Optional external video evaluation

This evaluates the existing image-trained classifier on clip-level majority votes from sampled RAVDESS frames. It does not train on RAVDESS. Keep the sample small for a quick pilot; use more actor archives and actor-disjoint evaluation before making generalization claims.


In [ ]:
# Optional: set RUN_RAVDESS_EVALUATION=True to score a small, balanced clip sample.
RUN_RAVDESS_EVALUATION = False
RAVDESS_CLIPS_PER_CLASS = 2  # 0 = every compatible clip; otherwise deterministic cap per class

def evaluate_ravdess(records=RAVDESS_RECORDS, clips_per_class=RAVDESS_CLIPS_PER_CLASS,
                     sample_fps=RAVDESS_SAMPLE_FPS, max_frames_per_clip=12):
    if not records:
        print("No compatible RAVDESS clips are available.")
        return None
    groups = {}
    for row in records:
        groups.setdefault(row["label"], []).append(row)
    selected = []
    for label in sorted(groups):
        group = groups[label]
        if clips_per_class and clips_per_class > 0 and len(group) > clips_per_class:
            ids = np.linspace(0, len(group) - 1, clips_per_class).round().astype(int)
            group = [group[i] for i in ids]
        selected.extend(group)

    y_true, y_pred, failures = [], [], []
    for row in selected:
        try:
            result = predict_video(str(row["path"]), predictor, sample_fps=sample_fps,
                                   max_samples=max_frames_per_clip, show=False, verbose=False)
            if result["summary"] is None:
                failures.append((row["filename"], result["status"]))
                continue
            y_true.append(row["label"])
            y_pred.append(result["summary"]["most_frequent_emotion"])
        except Exception as exc:
            failures.append((row["filename"], f"{type(exc).__name__}: {exc}"))

    print(f"Evaluated clips: {len(y_true)} / {len(selected)} selected; unusable/error clips: {len(failures)}")
    if failures:
        print("First skipped clip:", failures[0])
    if not y_true:
        print("No clip produced a face prediction; check the video codec and FACE_DETECTION settings.")
        return None

    labels = sorted(set(y_true))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    print(f"Clip accuracy: {accuracy_score(y_true, y_pred):.3f} | balanced accuracy: {balanced_accuracy_score(y_true, y_pred):.3f} | "
          f"macro-F1: {f1_score(y_true, y_pred, labels=labels, average='macro', zero_division=0):.3f}")
    print(classification_report(y_true, y_pred, labels=labels, target_names=labels, digits=3, zero_division=0))
    ConfusionMatrixDisplay(cm, display_labels=labels).plot(cmap="Blues", colorbar=False, xticks_rotation=45)
    plt.title("RAVDESS external pilot (clip-level majority vote)")
    plt.tight_layout(); plt.show()
    print("These figures describe only the selected acted clips/actors and are not a general-population emotion measure.")
    return {"y_true": y_true, "y_pred": y_pred, "confusion_matrix": cm.tolist(), "failures": failures}

if RUN_RAVDESS_EVALUATION:
    RAVDESS_EVALUATION = evaluate_ravdess()
else:
    print("RAVDESS benchmark is off by default; set RUN_RAVDESS_EVALUATION=True to run the pilot.")


In [ ]:
# 18. Try it on one video (uses a compatible RAVDESS clip automatically when available)
if SAMPLE_VIDEO_PATH:
    sample_video = SAMPLE_VIDEO_PATH
elif RAVDESS_SAMPLE_VIDEO:
    sample_video = RAVDESS_SAMPLE_VIDEO
    print("Using a video-only RAVDESS speech clip:", sample_video)
else:
    sample_video = pick_media(None, "video")
if sample_video:
    video_result = predict_video(sample_video, predictor, sample_fps=2.0)


`predict_video` options: `sample_fps` (samples per second of video, default 2), or `frame_step` to force every Nth frame; `max_samples` to cap the work on long videos; `all_faces=True` to classify every face in a frame instead of the largest. The result dict has `status` (`ok`, `no_faces_detected`, `no_readable_frames`), `video_info`, per-frame `frame_predictions` and a `summary`.

## Reload check

Rebuilds the predictor from the saved files only (as a separate session would) and confirms it reproduces the in-memory model.

In [ ]:
# 19. Reload from disk and compare
reloaded = keras.models.load_model(MODEL_PATH, compile=False)
saved_classes = json.loads(CLASS_MAP_PATH.read_text())["class_names"]
saved_cfg = json.loads(PREPROC_PATH.read_text())
assert saved_classes == CLASS_NAMES, "saved class order differs from the training class order"

xb_test, _ = next(iter(test_ds))
diff = float(np.abs(model(xb_test, training=False).numpy() - reloaded(xb_test, training=False).numpy()).max())
RELOAD_OK = diff < 1e-5
print("Reloaded classes:", saved_classes)
print(f"Max score difference, original vs reloaded model: {diff:.2e}  ->  {'OK' if RELOAD_OK else 'MISMATCH'}")

predictor_reloaded = Predictor(reloaded, saved_classes, saved_cfg, FACE_DETECTION)
if sample_image:
    _ = predict_image(sample_image, predictor_reloaded, show=False)

## Limitations (read before using the results)

- **Dataset labels are noisy.** FER-style datasets contain mislabelled and ambiguous faces; accuracy well below 100% is expected, and some classes are inherently confused (for example fear vs surprise, sad vs neutral).
- **Class imbalance.** `disgust` has far fewer images than the other classes. Class weights help, but check per-class recall, not just accuracy.
- **Small, low-resolution, mostly frontal faces.** Performance on real photos or video (lighting, angle, occlusion, glasses, different age/ethnicity mix) can be much worse than on the test split.
- **Scores are not probabilities of being correct.** The softmax output is uncalibrated and often over-confident.
- **Face detection is basic.** The Haar cascade finds frontal faces only and can miss or add faces; when no face is found in an image, the whole image is classified and the result says so.
- **A video result is a summary of sampled frames**, not a judgement of what a person felt; there is no tracking of identities and no audio.
- **Small datasets and BatchNorm.** With few training steps per epoch, BatchNorm's moving statistics (momentum 0.99) can lag behind the weights, so inference-mode accuracy can collapse to chance even though training accuracy is high. The check after training warns about this. Remedies: more data or epochs, or `BN_MOMENTUM = 0.9`.
- **Model design.** This is a small CNN trained from scratch; transfer learning would likely do better. Facial-expression models infer a visual expression, not an internal emotional state, and should not be used for decisions about people.
- Seeds are fixed, but GPU training is not guaranteed to be bit-for-bit reproducible.
- **RAVDESS is acted video, not a replacement for FER image training.** The notebook reads only video-only speech clips and skips emotion labels that do not map to the trained model. The two downloaded actors (05 and 06) are enough for a demo/pilot, not a representative evaluation. RAVDESS is CC BY-NC-SA 4.0; cite the dataset and follow its license if you share results.
- **Dataset mismatch matters.** A model trained on FER-style still images is evaluated on sampled RAVDESS video frames here; this is an external-domain check, not a comparable test-set score.


In [ ]:
# 20. Run checklist (computed from this session's state)
def mark(ok): return "[x]" if ok else "[ ]"
checks = [
    (True, "Dataset located and class folders identical across splits"),
    (len(all_classes) == NUM_CLASSES, "All dataset classes used (no silent omission)") if not CLASSES_TO_USE
        else (True, f"Classes deliberately restricted to {CLASS_NAMES}"),
    (model.output_shape[-1] == NUM_CLASSES, f"Model output size ({model.output_shape[-1]}) == number of classes ({NUM_CLASSES})"),
    (True, f"Validation data: {VAL_SOURCE}"),
    (not (set(map(str, test_paths)) & set(map(str, fit_paths + val_paths))), "Test split disjoint from train/validation and used only for the final evaluation"),
    ("loss" in hist, "Training/validation curves plotted"),
    ("accuracy" in report, "Test accuracy, per-class report and confusion matrix produced"),
    (all(p.exists() for p in (MODEL_PATH, CLASS_MAP_PATH, PREPROC_PATH)), "Model + class mapping + preprocessing config saved"),
    (RELOAD_OK, "Reloaded artifacts reproduce the in-memory model"),
    (bool(sample_image), "Image prediction example run"),
    (bool(sample_video), "Video prediction example used a configured/uploaded video or a compatible RAVDESS clip"),
    (bool(RAVDESS_RECORDS), "RAVDESS video-only speech clips found/extracted (optional)"),
]
for ok, text in checks:
    print(mark(ok), text)